In [ ]:
import base64
import io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pypdfium2 as pdfium

ENDPOINT = "http://localhost:8000/v1/chat/completions"
MODEL = "lightonai/LightOnOCR-2-1B"

# Local PDF path
pdf_path = Path("../data/inputs/arxiv/pdfs/2303.13740.pdf")

# Render all pages to base64 PNGs
pdf = pdfium.PdfDocument(str(pdf_path))
page_images_b64 = []
for i in range(len(pdf)):
    page = pdf[i]
    # Render at 200 DPI (scale factor = 200/72 ≈ 2.77)
    pil_image = page.render(scale=2.77).to_pil()
    buffer = io.BytesIO()
    pil_image.save(buffer, format="PNG")
    page_images_b64.append(base64.b64encode(buffer.getvalue()).decode("utf-8"))

# Build one chat payload per page
payloads = []
for image_base64 in page_images_b64:
    payloads.append({
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": [{
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_base64}"}
            }]
        }],
        "max_tokens": 4096,
        "temperature": 0.2,
        "top_p": 0.9,
    })

# vLLM's OpenAI-compatible chat endpoint does NOT accept a list of requests
# in a single POST. "Batching" is achieved by sending multiple requests in
# parallel (the server will batch internally).

def post_one(idx_payload):
    idx, payload = idx_payload
    r = requests.post(ENDPOINT, json=payload)
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    return idx, content

# Tune max_workers to your GPU/CPU and server capacity
max_workers = 8
page_texts = [None] * len(payloads)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(post_one, ip) for ip in enumerate(payloads)]
    for fut in as_completed(futures):
        idx, content = fut.result()
        page_texts[idx] = content

# page_texts now has OCR output per page, in order
print(f"Pages: {len(page_texts)}")
print(page_texts[0][:500])
